# Silver Technical Indicators

In [0]:
%run ../config/config

In [0]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, DateType, DoubleType

#### 1. Read cleaned data

In [0]:
cleaned_df = spark.table(tables["cleaned_ohlcv"])
print(f"Input rows: {cleaned_df.count():,}")

#### 2. Compute indicators

In [0]:
indicator_schema = StructType([
    StructField("symbol", StringType(), False),
    StructField("date", DateType(), False),
    StructField("rsi_14", DoubleType()),
    StructField("rsi_28", DoubleType()),
    StructField("macd", DoubleType()),
    StructField("macd_signal", DoubleType()),
    StructField("macd_hist", DoubleType()),
    StructField("ema_12", DoubleType()),
    StructField("ema_26", DoubleType()),
    StructField("sma_20", DoubleType()),
    StructField("sma_50", DoubleType()),
    StructField("bb_upper", DoubleType()),
    StructField("bb_middle", DoubleType()),
    StructField("bb_lower", DoubleType()),
    StructField("bb_width", DoubleType()),
    StructField("bb_pct", DoubleType()),
    StructField("obv", DoubleType()),
    StructField("vwap_20", DoubleType()),
    StructField("volume_ratio", DoubleType()),
    StructField("atr_14", DoubleType()),
    StructField("adx_14", DoubleType()),
    StructField("plus_di", DoubleType()),
    StructField("minus_di", DoubleType()),
    StructField("stoch_k", DoubleType()),
    StructField("stoch_d", DoubleType()),
])


def compute_rsi(series: pd.Series, period: int) -> pd.Series:
    """Compute RSI using moving average of gains/losses."""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def compute_atr(high: pd.Series, low: pd.Series, close: pd.Series, period: int) -> pd.Series:
    """Compute Average True Range."""
    prev_close = close.shift(1)
    tr1 = high - low
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    return tr.ewm(span=period, min_periods=period, adjust=False).mean()


def compute_adx(high: pd.Series, low: pd.Series, close: pd.Series, period: int = 14, atr=None):
    """Compute ADX, +DI, -DI."""
    prev_high = high.shift(1)
    prev_low = low.shift(1)

    plus_dm = (high - prev_high).clip(lower=0)
    minus_dm = (prev_low - low).clip(lower=0)

    plus_dm[plus_dm < minus_dm] = 0
    minus_dm[minus_dm < plus_dm] = 0

    if atr is None:
        atr = compute_atr(high, low, close, period)
    atr = atr.replace(0, np.nan)

    plus_di = 100 * plus_dm.ewm(span=period, min_periods=period, adjust=False).mean() / atr
    minus_di = 100 * minus_dm.ewm(span=period, min_periods=period, adjust=False).mean() / atr

    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)
    adx = dx.ewm(span=period, min_periods=period, adjust=False).mean()

    return adx, plus_di, minus_di


def compute_indicators(pdf: pd.DataFrame) -> pd.DataFrame:
    """Compute all technical indicators for one symbol's data."""
    pdf = pdf.sort_values("date").reset_index(drop=True)

    close = pdf["close"]
    high = pdf["high"]
    low = pdf["low"]
    volume = pdf["volume"].astype(float)
    typical = pdf["typical_price"] if "typical_price" in pdf.columns else (high + low + close) / 3

    result = pd.DataFrame()
    result["symbol"] = pdf["symbol"]
    result["date"] = pdf["date"]

    # RSI 
    result["rsi_14"] = compute_rsi(close, 14).round(2)
    result["rsi_28"] = compute_rsi(close, 28).round(2)

    # MACD 
    ema_12 = close.ewm(span=12, adjust=False).mean()
    ema_26 = close.ewm(span=26, adjust=False).mean()
    macd_line = ema_12 - ema_26
    macd_signal = macd_line.ewm(span=9, adjust=False).mean()

    result["macd"] = macd_line.round(2)
    result["macd_signal"] = macd_signal.round(2)
    result["macd_hist"] = (macd_line - macd_signal).round(2)
    result["ema_12"] = ema_12.round(2)
    result["ema_26"] = ema_26.round(2)

    # SMA 
    result["sma_20"] = close.rolling(20).mean().round(2)
    result["sma_50"] = close.rolling(50).mean().round(2)

    # Bollinger Bands 
    bb_middle = result["sma_20"]
    bb_std = close.rolling(20).std()
    result["bb_upper"] = (bb_middle + 2 * bb_std).round(2)
    result["bb_middle"] = bb_middle
    result["bb_lower"] = (bb_middle - 2 * bb_std).round(2)
    result["bb_width"] = ((result["bb_upper"] - result["bb_lower"]) / bb_middle.replace(0, np.nan)).round(4)
    result["bb_pct"] = ((close - result["bb_lower"]) / (result["bb_upper"] - result["bb_lower"]).replace(0, np.nan)).round(4)

    # OBV 
    obv_direction = np.sign(close.diff()).fillna(0).astype(int)
    result["obv"] = (obv_direction * volume).cumsum().round(0)

    # VWAP 
    cum_vol_price = (typical * volume).rolling(20).sum()
    cum_vol = volume.rolling(20).sum()
    result["vwap_20"] = (cum_vol_price / cum_vol.replace(0, np.nan)).round(2)

    # Volume Ratio 
    vol_sma_20 = volume.rolling(20).mean()
    result["volume_ratio"] = (volume / vol_sma_20.replace(0, np.nan)).round(2)

    # ATR 
    atr_14 = compute_atr(high, low, close, 14)
    result["atr_14"] = atr_14.round(2)

    # ADX 
    adx, plus_di, minus_di = compute_adx(high, low, close, 14, atr=atr_14)
    result["adx_14"] = adx.round(2)
    result["plus_di"] = plus_di.round(2)
    result["minus_di"] = minus_di.round(2)

    # Stochastic Oscillator 
    lowest_14 = low.rolling(14).min()
    highest_14 = high.rolling(14).max()
    stoch_k = 100 * (close - lowest_14) / (highest_14 - lowest_14).replace(0, np.nan)
    result["stoch_k"] = stoch_k.round(2)
    result["stoch_d"] = stoch_k.rolling(3).mean().round(2)

    return result

#### 3. Apply to all symbols

In [0]:
input_df = cleaned_df.select("symbol", "date", "open", "high", "low", "close", "volume", "typical_price")

# aapply each symbol
indicators_df = input_df.groupBy("symbol").applyInPandas(compute_indicators, schema=indicator_schema)

indicators_df = indicators_df.cache()

#### 4. Checks

In [0]:
# RSI check
rsi_check = indicators_df.filter(
    (F.col("rsi_14") < 0) | (F.col("rsi_14") > 100)
).count()
print(f"RSI out of range: {rsi_check} rows")

# BB order check: lower <= middle <= upper
bb_check = indicators_df.filter(
    (F.col("bb_lower") > F.col("bb_middle")) |
    (F.col("bb_middle") > F.col("bb_upper"))
).count()
print(f"BB ordering violated: {bb_check} rows")

#### 5. Write to Silver

In [0]:
indicators_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tables["technical_indicators"])
indicators_df.unpersist()
print(f"Successfull written to {tables['technical_indicators']}")

#### 6. Verify

In [0]:
%sql
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT symbol) AS unique_symbols
FROM wyckoff_catalog.silver.technical_indicators

In [0]:
%sql
SELECT * FROM wyckoff_catalog.silver.technical_indicators
WHERE symbol = 'HPG'
ORDER BY date DESC
LIMIT 20